# Firmware Supply-Chain PoC — Colab Review
**Complete example:** Trusted baseline → harmless manipulation → generated evidence → deterministic result → explanation.

Use the existing `before-sir-corrections` repository branch. No malware and no real OT action are involved.

In [1]:
!git clone -b before-sir-corrections https://github.com/NiharR777/Firmware-Supply-Chain-PoC.git /content/Firmware-Supply-Chain-PoC
%cd /content/Firmware-Supply-Chain-PoC
!git status
!git log -1 --oneline


Cloning into '/content/Firmware-Supply-Chain-PoC'...
remote: Enumerating objects: 679, done.
remote: Counting objects: 100% (679/679), done.
remote: Compressing objects: 100% (330/330), done.
remote: Total 679 (delta 357), reused 644 (delta 323), pack-reused 0 (from 0)
Receiving objects: 100% (679/679), 3.63 MiB | 13.24 MiB/s, done.
Resolving deltas: 100% (357/357), done.
/content/Firmware-Supply-Chain-PoC
On branch before-sir-corrections
Your branch is up to date with 'origin/before-sir-corrections'.

nothing to commit, working tree clean
f328fbe (HEAD -> before-sir-corrections, origin/before-sir-corrections) Apply correction review and validation updates


## 1. Existing correction validation

In [2]:
!python scripts/run_correction_checks.py --output /content/correction-checks


[PASS] -m pytest -q
[PASS] scripts/validate_dataset.py
[PASS] scripts/validate_grouped_split.py
[PASS] scripts/validate_schemas.py
[PASS] scripts/verify_evidence_bundle.py
[PASS] scripts/validate_correction.py
Logs: /content/correction-checks/correction-checks-yellcwv1
Experiment v1 remains REJECTED regardless of package validation outcome.


## 2. SC-001: inspect the existing evidence bundle

In [3]:
from pathlib import Path
import json, hashlib

ROOT=Path("/content/Firmware-Supply-Chain-PoC")
BUNDLE=ROOT/"data/synthetic/supply_chain_poc/generated/evidence_bundles/SC-001"

print("SC-001 files:")
for p in sorted(BUNDLE.iterdir()):
    print(" -",p.name)

def readj(name):
    with open(BUNDLE/name,encoding="utf-8") as f: return json.load(f)

hash_evidence=readj("hash_evidence.json")
decision=readj("decision.json")

print("\n=== HASH EVIDENCE ===")
print(json.dumps(hash_evidence,indent=2))
print("\n=== DETERMINISTIC DECISION ===")
print(json.dumps(decision,indent=2))


SC-001 files:
 - asset_mapping.json
 - decision.json
 - freshness_evidence.json
 - hash_evidence.json
 - package.json
 - reference_evidence.json
 - rollback_evidence.json
 - sbom.json
 - signature_evidence.json
 - vendor_evidence.json

=== HASH EVIDENCE ===
{
  "package_id": "PKG-001",
  "hash_algorithm": "SHA-256",
  "observed_hash": "28073b6f663b1af06c5842ff5d62022770c4f4c4834e1cc9dcca8d6c3d5bb3ed",
  "hash_match": false,
  "observed_package_path": "data/synthetic/supply_chain_poc/generated/fixtures/modified/SC-001_PKG-001_modified.bin",
  "observed_package_size_bytes": 122,
  "calculation_timestamp": "2026-09-01T11:15:41+00:00",
  "integrity_status": "FAILED",
  "data_provenance": "SYNTHETIC OBSERVED EVIDENCE"
}

=== DETERMINISTIC DECISION ===
{
  "scenario_id": "SC-001",
  "package_id": "PKG-001",
  "risk_score": 40,
  "risk_level": "high_risk",
  "risk_factors": [
    "hash_integrity_failure"
  ],
  "risk_breakdown": [
    {
      "factor": "hash_integrity_failure",
      "weight"

## 3. Verify the hash evidence from the actual files

The corrected specification requires SC-001 to use genuinely different SHA-256 values from harmless files, with the mismatch derived from `expected_hash != observed_hash` rather than from the scenario label.

In [4]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

print("expected_hash:",hash_evidence.get("expected_hash"))
print("observed_hash:",hash_evidence.get("observed_hash",hash_evidence.get("calculated_hash")))
print("hash_match:",hash_evidence.get("hash_match",hash_evidence.get("hash_status")))
print("decision:",decision.get("risk_level",decision.get("decision")))
print("recommendation:",decision.get("recommendation"))
print("deployment_allowed:",decision.get("deployment_allowed"))
print("human_approval_required:",decision.get("human_approval_required"))
print("real_action_executed:",decision.get("real_action_executed"))


expected_hash: None
observed_hash: 28073b6f663b1af06c5842ff5d62022770c4f4c4834e1cc9dcca8d6c3d5bb3ed
hash_match: False
decision: high_risk
recommendation: quarantine_review
deployment_allowed: False
human_approval_required: True
real_action_executed: False


## 4. ML dataset contract — do not retrain yet


In [5]:
import pandas as pd
DATA=ROOT/"data/synthetic/supply_chain_poc/dataset"
features=pd.read_csv(DATA/"engineered_features.csv")
labels=pd.read_csv(DATA/"ground_truth_labels.csv")
splits=pd.read_csv(DATA/"split_assignments.csv")

print("Features:",features.shape)
print("Labels:",labels.shape)
print("Splits:",splits.shape)
print("\nLabels:")
print(labels["ground_truth_label"].value_counts())
print("\nSplits:")
print(splits["split"].value_counts())


Features: (5000, 23)
Labels: (5000, 4)
Splits: (5000, 5)

Labels:
ground_truth_label
high_risk      2000
investigate    2000
normal         1000
Name: count, dtype: int64

Splits:
split
train         3500
test           750
validation     750
Name: count, dtype: int64


## 5. Experiment v2 — approved Isolation Forest search space
**This cell only prints the approved configuration.**

In [6]:
from itertools import product

cfg={
"n_estimators":[300,600],
"max_samples":["auto",0.75],
"max_features":[0.75,1.0],
"bootstrap":[False],
"contamination":["auto"],
"random_state":[20260828],
}
keys=list(cfg)
configs=[dict(zip(keys,v)) for v in product(*[cfg[k] for k in keys])]
print("Approved configurations:",len(configs))
for i,c in enumerate(configs,1): print(i,c)


Approved configurations: 8
1 {'n_estimators': 300, 'max_samples': 'auto', 'max_features': 0.75, 'bootstrap': False, 'contamination': 'auto', 'random_state': 20260828}
2 {'n_estimators': 300, 'max_samples': 'auto', 'max_features': 1.0, 'bootstrap': False, 'contamination': 'auto', 'random_state': 20260828}
3 {'n_estimators': 300, 'max_samples': 0.75, 'max_features': 0.75, 'bootstrap': False, 'contamination': 'auto', 'random_state': 20260828}
4 {'n_estimators': 300, 'max_samples': 0.75, 'max_features': 1.0, 'bootstrap': False, 'contamination': 'auto', 'random_state': 20260828}
5 {'n_estimators': 600, 'max_samples': 'auto', 'max_features': 0.75, 'bootstrap': False, 'contamination': 'auto', 'random_state': 20260828}
6 {'n_estimators': 600, 'max_samples': 'auto', 'max_features': 1.0, 'bootstrap': False, 'contamination': 'auto', 'random_state': 20260828}
7 {'n_estimators': 600, 'max_samples': 0.75, 'max_features': 0.75, 'bootstrap': False, 'contamination': 'auto', 'random_state': 20260828}
8 

In [7]:
# Find the trusted/reference hash for SC-001 / PKG-001
import os
import json
from pathlib import Path

ROOT = Path("/content/Firmware-Supply-Chain-PoC")

matches = []

for path in ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in [".json", ".csv"]:
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
            if "PKG-001" in text or "28073b6f663b1af06c5842ff5d62022770c4f4c4834e1cc9dcca8d6c3d5bb3ed" in text:
                matches.append(path)
        except Exception:
            pass

print("Files containing PKG-001 or the observed hash:")
for p in matches:
    print(p.relative_to(ROOT))

Files containing PKG-001 or the observed hash:
package_manifest.json
data/synthetic/supply_chain_poc/scenario_catalog.csv
data/synthetic/supply_chain_poc/deployments/PKG-001_deployment.json
data/synthetic/supply_chain_poc/sbom/PKG-001_sbom.json
data/synthetic/supply_chain_poc/generated/generated_package_manifest.csv
data/synthetic/supply_chain_poc/signatures/PKG-001_signature.json
data/synthetic/supply_chain_poc/dataset/scenario_manifests.csv
data/synthetic/supply_chain_poc/dataset/split_assignments.csv
data/synthetic/supply_chain_poc/dataset/observable_features.csv
data/synthetic/supply_chain_poc/dataset/reference_data.csv
data/synthetic/supply_chain_poc/schemas/dependencies.csv
data/synthetic/supply_chain_poc/schemas/packages.csv
data/synthetic/supply_chain_poc/schemas/package_evidence.csv
data/synthetic/supply_chain_poc/schemas/vex_records.csv
data/synthetic/supply_chain_poc/generated/scenarios/SC-005.json
data/synthetic/supply_chain_poc/generated/scenarios/SC-001.json
data/syntheti

In [8]:
import pandas as pd
import json
from pathlib import Path

ROOT = Path("/content/Firmware-Supply-Chain-PoC")

# 1. Reference data
reference = pd.read_csv(
    ROOT / "data/synthetic/supply_chain_poc/dataset/reference_data.csv"
)

print("=== REFERENCE DATA ===")
print(reference.columns.tolist())
print(reference[reference.astype(str).apply(
    lambda row: row.str.contains("PKG-001", case=False, na=False).any(), axis=1
)].to_string(index=False))


# 2. Package manifest
manifest = pd.read_csv(
    ROOT / "data/synthetic/supply_chain_poc/dataset/package_manifest.csv"
)

print("\n=== PACKAGE MANIFEST ===")
print(manifest.columns.tolist())
print(manifest[manifest.astype(str).apply(
    lambda row: row.str.contains("PKG-001", case=False, na=False).any(), axis=1
)].to_string(index=False))


# 3. Reference evidence
ref_path = (
    ROOT /
    "data/synthetic/supply_chain_poc/generated/evidence_bundles/SC-001/reference_evidence.json"
)

print("\n=== REFERENCE EVIDENCE ===")
with open(ref_path, encoding="utf-8") as f:
    reference_evidence = json.load(f)

print(json.dumps(reference_evidence, indent=2))

=== REFERENCE DATA ===
['record_id', 'package_id', 'lineage_id', 'approved_vendor_id', 'trusted_hash', 'approved_version', 'freshness_threshold_days', 'approved_component_count', 'reference_profile_version']
record_id  package_id   lineage_id approved_vendor_id                                                     trusted_hash approved_version  freshness_threshold_days  approved_component_count reference_profile_version
REC-00091 PKG-0010-01 LINEAGE-0010           VEND-001 68ed8d48ed11e97554bc77a8393bcdfbd43d6dbfd2a9123d02717f4df7e40c17            6.7.9                        30                        11                 REF-2.0.0
REC-00092 PKG-0010-02 LINEAGE-0010           VEND-001 0206687d93dc07351d9c011305096f69316126a12a678a9e1d9aef534a974c21            6.7.9                        30                        11                 REF-2.0.0
REC-00093 PKG-0010-03 LINEAGE-0010           VEND-001 91d6d77beb8833d8e2a76c91a4beb1cc07276bcc289c866d77419dd80fcfae5b            6.7.9               

FileNotFoundError: [Errno 2] No such file or directory: '/content/Firmware-Supply-Chain-PoC/data/synthetic/supply_chain_poc/dataset/package_manifest.csv'

In [9]:
import json
from pathlib import Path

ROOT = Path("/content/Firmware-Supply-Chain-PoC")

files_to_check = [
    ROOT/"data/synthetic/supply_chain_poc/package_manifest.json",
    ROOT/"data/synthetic/supply_chain_poc/generated/package_manifest.csv",
    ROOT/"data/synthetic/supply_chain_poc/sbom/PKG-001_sbom.json",
]

for f in files_to_check:
    print(f"\n===== {f.name} =====")
    if f.exists():
        print(f.read_text()[:5000])
    else:
        print("NOT FOUND")


===== package_manifest.json =====
NOT FOUND

===== package_manifest.csv =====
NOT FOUND

===== PKG-001_sbom.json =====
{
  "bomFormat": "CycloneDX",
  "specVersion": "1.5",
  "serialNumber": "urn:uuid:e32494ed-d3f1-537e-82dd-d684cc352a5b",
  "version": 1,
  "metadata": {
    "component": {
      "type": "application",
      "name": "PKG-001",
      "version": "2.4.1"
    }
  },
  "components": [
    {
      "type": "library",
      "bom-ref": "pkg:generic/openssl@3.0.12",
      "name": "OpenSSL",
      "version": "3.0.12"
    },
    {
      "type": "library",
      "bom-ref": "pkg:generic/curl@8.4.0",
      "name": "cURL",
      "version": "8.4.0"
    },
    {
      "type": "library",
      "bom-ref": "pkg:generic/zlib@1.3",
      "name": "zlib",
      "version": "1.3"
    }
  ],
  "vulnerabilities": []
}


In [10]:
import json
from pathlib import Path

ROOT = Path("/content/Firmware-Supply-Chain-PoC")

for path in ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in [".json", ".csv"]:
        try:
            text = path.read_text(errors="ignore")
            if "PKG-001" in text and "hash" in text.lower():
                print("\nFOUND:", path.relative_to(ROOT))
                print(text[:3000])
        except:
            pass


FOUND: package_manifest.json
{
  "experiment_v1_status": "REJECTED",
  "files": {
    ".gitattributes": "8ed1fb9bdd0419d0bdf4306536e4ad53fc9694bf2fb7ba7b6225ba88f16edf92",
    ".gitignore": "7c1ec85a4f97f5d4028181c4ad2f798fbd75e8d998b4c2edd39a136dfc2fc2c6",
    "CORRECTION_SUMMARY.md": "5d3c3dbea7a01290dfd51cca7542942bb6c88f0d92b61f84260ffee90f1911e4",
    "README.md": "3dc89a1c222e1697e09b90c5606002e59fa709a0e740eeb0d55f5f4900c48306",
    "VALIDATION_RESULTS.md": "bd77bd638ab2bdcdd769bce7fe799bbb2e537d147916a6a91889907a94e66787",
    "artifacts/correction/calibration_report.json": "d2cded80273a4a1ee07b691fa12910f51c399870d0f5fe468319c157bdc656ae",
    "artifacts/correction/environment_record.json": "af5bcf5126230a2db0074ed880cc8162e22899410e622e44dcb06f96fe9a83df",
    "artifacts/correction/explanation_report.json": "15b9b4095e38c76f56612c01b97c827fb9526f631b3e3050a8be067ba101b249",
    "artifacts/correction/input_profile.json": "244bb1c43b2523de81f4d55c1de7b29f9ac260729107e1eeb4fd6d

In [11]:
from pathlib import Path

ROOT = Path("/content/Firmware-Supply-Chain-PoC")

for p in ROOT.rglob("*PKG-001*"):
    if p.is_file():
        print(p.relative_to(ROOT))

data/synthetic/supply_chain_poc/deployments/PKG-001_deployment.json
data/synthetic/supply_chain_poc/sbom/PKG-001_sbom.json
data/synthetic/supply_chain_poc/signatures/PKG-001_signature.json
data/synthetic/supply_chain_poc/packages/PKG-001_2.4.1.bin
data/synthetic/supply_chain_poc/generated/fixtures/modified/SC-001_PKG-001_modified.bin
data/synthetic/supply_chain_poc/generated/fixtures/clean/PKG-001_2.4.1_clean.bin


In [12]:
from pathlib import Path
import json

ROOT = Path("/content/Firmware-Supply-Chain-PoC")

files = [
    ROOT/"data/synthetic/supply_chain_poc/packages/PKG-001_2.4.1.bin",
    ROOT/"data/synthetic/supply_chain_poc/sbom/PKG-001_sbom.json",
    ROOT/"data/synthetic/supply_chain_poc/package_manifests/PKG-001_package.json",
    ROOT/"data/synthetic/supply_chain_poc/deployments/PKG-001_deployment.json",
]

for f in files:
    print("\n==============================")
    print(f.name)
    print("==============================")
    if f.suffix == ".json":
        print(json.dumps(json.loads(f.read_text()), indent=2))
    else:
        print("Binary package:", f.stat().st_size, "bytes")


PKG-001_2.4.1.bin
Binary package: 188 bytes

PKG-001_sbom.json
{
  "bomFormat": "CycloneDX",
  "specVersion": "1.5",
  "serialNumber": "urn:uuid:e32494ed-d3f1-537e-82dd-d684cc352a5b",
  "version": 1,
  "metadata": {
    "component": {
      "type": "application",
      "name": "PKG-001",
      "version": "2.4.1"
    }
  },
  "components": [
    {
      "type": "library",
      "bom-ref": "pkg:generic/openssl@3.0.12",
      "name": "OpenSSL",
      "version": "3.0.12"
    },
    {
      "type": "library",
      "bom-ref": "pkg:generic/curl@8.4.0",
      "name": "cURL",
      "version": "8.4.0"
    },
    {
      "type": "library",
      "bom-ref": "pkg:generic/zlib@1.3",
      "name": "zlib",
      "version": "1.3"
    }
  ],
  "vulnerabilities": []
}

PKG-001_package.json


FileNotFoundError: [Errno 2] No such file or directory: '/content/Firmware-Supply-Chain-PoC/data/synthetic/supply_chain_poc/package_manifests/PKG-001_package.json'

In [13]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/Firmware-Supply-Chain-PoC")

# Actual files from the repository
trusted_file = ROOT / "data/synthetic/supply_chain_poc/generated/fixtures/clean/PKG-001_2.4.1_clean.bin"
modified_file = ROOT / "data/synthetic/supply_chain_poc/generated/fixtures/modified/SC-001_PKG-001_modified.bin"

sbom_file = ROOT / "data/synthetic/supply_chain_poc/sbom/PKG-001_sbom.json"
signature_file = ROOT / "data/synthetic/supply_chain_poc/signatures/PKG-001_signature.json"

decision_file = ROOT / "data/synthetic/supply_chain_poc/generated/evidence_bundles/SC-001/decision.json"
hash_evidence_file = ROOT / "data/synthetic/supply_chain_poc/generated/evidence_bundles/SC-001/hash_evidence.json"


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


print("=" * 70)
print("SC-001 COMPLETE VALIDATION")
print("=" * 70)

# --------------------------------------------------
# 1. TRUSTED BASELINE
# --------------------------------------------------

print("\n[1] TRUSTED BASELINE")

print("File:", trusted_file)
print("Exists:", trusted_file.exists())

if trusted_file.exists():
    trusted_hash = sha256(trusted_file)
    print("Trusted SHA-256:", trusted_hash)
    print("Size:", trusted_file.stat().st_size, "bytes")


# --------------------------------------------------
# 2. MANIPULATED / OBSERVED PACKAGE
# --------------------------------------------------

print("\n[2] MANIPULATED PACKAGE")

print("File:", modified_file)
print("Exists:", modified_file.exists())

if modified_file.exists():
    observed_hash = sha256(modified_file)
    print("Observed SHA-256:", observed_hash)
    print("Size:", modified_file.stat().st_size, "bytes")


# --------------------------------------------------
# 3. HASH COMPARISON
# --------------------------------------------------

print("\n[3] INTEGRITY COMPARISON")

if trusted_file.exists() and modified_file.exists():

    hash_match = trusted_hash == observed_hash

    print("Expected hash :", trusted_hash)
    print("Observed hash :", observed_hash)
    print("Hash match    :", hash_match)

    if hash_match:
        print("Result: TRUSTED")
    else:
        print("Result: HASH MISMATCH → ANOMALY")


# --------------------------------------------------
# 4. SBOM
# --------------------------------------------------

print("\n[4] SBOM")

print("File:", sbom_file)
print("Exists:", sbom_file.exists())

if sbom_file.exists():
    sbom = json.loads(sbom_file.read_text())

    print("Format:", sbom.get("bomFormat"))
    print("Specification:", sbom.get("specVersion"))

    component = sbom.get("metadata", {}).get("component", {})

    print("Package:", component.get("name"))
    print("Version:", component.get("version"))

    print("\nDependencies:")
    for component in sbom.get("components", []):
        print(
            "-",
            component.get("name"),
            component.get("version")
        )


# --------------------------------------------------
# 5. GENERATED HASH EVIDENCE
# --------------------------------------------------

print("\n[5] GENERATED EVIDENCE")

if hash_evidence_file.exists():

    hash_evidence = json.loads(hash_evidence_file.read_text())

    print(json.dumps(hash_evidence, indent=2))


# --------------------------------------------------
# 6. DETERMINISTIC DECISION
# --------------------------------------------------

print("\n[6] DETERMINISTIC DECISION")

if decision_file.exists():

    decision = json.loads(decision_file.read_text())

    print("Risk score:", decision.get("risk_score"))
    print("Risk level:", decision.get("risk_level"))
    print("Risk factors:", decision.get("risk_factors"))
    print("Recommendation:", decision.get("recommendation"))
    print("Deployment allowed:", decision.get("deployment_allowed"))
    print("Human approval required:", decision.get("human_approval_required"))
    print("Real action executed:", decision.get("real_action_executed"))

    print("\nExplanation:")
    for reason in decision.get("reasons", []):
        print("-", reason)


print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

SC-001 COMPLETE VALIDATION

[1] TRUSTED BASELINE
File: /content/Firmware-Supply-Chain-PoC/data/synthetic/supply_chain_poc/generated/fixtures/clean/PKG-001_2.4.1_clean.bin
Exists: True
Trusted SHA-256: 2242089d86e08fb8e63ae20e008644f83f4dbf2f9ba014d3e284764bd715da5c
Size: 77 bytes

[2] MANIPULATED PACKAGE
File: /content/Firmware-Supply-Chain-PoC/data/synthetic/supply_chain_poc/generated/fixtures/modified/SC-001_PKG-001_modified.bin
Exists: True
Observed SHA-256: 28073b6f663b1af06c5842ff5d62022770c4f4c4834e1cc9dcca8d6c3d5bb3ed
Size: 122 bytes

[3] INTEGRITY COMPARISON
Expected hash : 2242089d86e08fb8e63ae20e008644f83f4dbf2f9ba014d3e284764bd715da5c
Observed hash : 28073b6f663b1af06c5842ff5d62022770c4f4c4834e1cc9dcca8d6c3d5bb3ed
Hash match    : False
Result: HASH MISMATCH → ANOMALY

[4] SBOM
File: /content/Firmware-Supply-Chain-PoC/data/synthetic/supply_chain_poc/sbom/PKG-001_sbom.json
Exists: True
Format: CycloneDX
Specification: 1.5
Package: PKG-001
Version: 2.4.1

Dependencies:
- OpenSS